[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rgarzonj/python_data_science_basics/blob/main/collab/from_spreadsheets_to_algorithms.ipynb)

> **Google Colab note:** all required packages (`numpy`, `pandas`, `matplotlib`, `seaborn`, `scikit-learn`, `ipywidgets`) are pre-installed in Colab — just run the cells in order.

# From Spreadsheets to Algorithms

**A 60-minute hands-on tour of the ideas that drive modern data-driven decision-making.**

---

### What you'll learn
By the end of this notebook, you will have personally:
1. **Loaded and explored** a realistic insurance dataset
2. **Visualized** patterns in claims, premiums, and customer risk
3. **Built a predictive model** that estimates insurance charges
4. **Segmented customers** into actionable risk groups
5. **Quantified the business value** of these techniques

### How to use this notebook
- Each cell of code can be run by clicking on it and pressing **Shift + Enter**
- You don't need to write any code — just run the cells in order
- **Interactive sliders** let you change inputs and see results update live
- Look for the 💡 **Executive Insight** boxes for the "so what?"

> *No prior coding experience required. The goal is intuition, not syntax.*

## 0. Setup — Loading the toolkit

Below we load the standard data science libraries. Think of these as specialized teams:

| Library | Role |
|---|---|
| `pandas` | The **analyst** — organizes data into tables |
| `numpy` | The **mathematician** — handles numerical computation |
| `matplotlib` & `seaborn` | The **designer** — creates charts |
| `scikit-learn` | The **modeler** — builds predictive algorithms |
| `ipywidgets` | The **interactive layer** — sliders and dropdowns |

Run the cell below (Shift + Enter) to import everything.

In [ ]:
# Standard data science stack
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, r2_score

# Interactivity
import ipywidgets as widgets
from IPython.display import display, Markdown, HTML

# Visual style
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

print("✅ Toolkit loaded successfully. You're ready to go.")

## 1. The Data — What does an insurance dataset look like?

We'll work with a realistic dataset of **2,000 health insurance policyholders**. Each row is one customer; each column is a fact about them.

In a real company, this data would come from your policy administration system, claims database, or CRM.

In [ ]:
# Generate a realistic synthetic insurance dataset
# (In a real engagement, this line would read from your company's database)
np.random.seed(42)
n = 2000

ages = np.random.randint(18, 75, n)
sex = np.random.choice(['male', 'female'], n)
bmi = np.clip(np.random.normal(28, 6, n), 16, 50).round(1)
children = np.random.choice([0, 1, 2, 3, 4], n, p=[0.42, 0.24, 0.20, 0.10, 0.04])
smoker = np.random.choice(['yes', 'no'], n, p=[0.20, 0.80])
region = np.random.choice(['northeast', 'southeast', 'southwest', 'northwest'], n)

# Charges follow a realistic pattern: age, BMI, and smoking status drive cost
charges = (
    250 * ages
    + 350 * (bmi - 25).clip(0, None)
    + 22000 * (smoker == 'yes')
    + 500 * children
    + np.random.normal(0, 2500, n)
).clip(1000, None).round(2)

df = pd.DataFrame({
    'age': ages,
    'sex': sex,
    'bmi': bmi,
    'children': children,
    'smoker': smoker,
    'region': region,
    'annual_charges': charges
})

print(f"Dataset shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print()
print("Here are the first 8 customers:")
df.head(8)

> 💡 **Executive Insight:** Every data science project starts here — with a *table*. Rows are the units you make decisions about (customers, claims, policies). Columns are the attributes you know about them. Most of the value comes from columns you *don't yet collect* but could.

## 2. Descriptive Analytics — *What happened?*

Before any fancy modeling, the first question is: **what does our book of business look like today?**

In [ ]:
# A one-line summary of every numeric column
summary = df.describe().round(2)
summary

**Reading this table:**
- The **average** customer is in their mid-40s with a BMI of ~28 and pays ~$15,000/year
- The **range** of charges is huge — from a few thousand to tens of thousands of dollars
- That spread is where the business problem (and the opportunity) lives

In [ ]:
# Visualize the distribution of annual charges
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['annual_charges'], bins=40, color='#2E86AB', edgecolor='white')
axes[0].set_title('How much do customers pay per year?')
axes[0].set_xlabel('Annual charges ($)')
axes[0].set_ylabel('Number of customers')
axes[0].axvline(df['annual_charges'].mean(), color='red', linestyle='--', label=f"Mean: ${df['annual_charges'].mean():,.0f}")
axes[0].axvline(df['annual_charges'].median(), color='orange', linestyle='--', label=f"Median: ${df['annual_charges'].median():,.0f}")
axes[0].legend()

# Smoker vs non-smoker — the single biggest driver
sns.boxplot(data=df, x='smoker', y='annual_charges', ax=axes[1], hue='smoker', palette=['#06A77D', '#D62246'], legend=False)
axes[1].set_title('Smoking status vs. annual charges')
axes[1].set_xlabel('Smoker')
axes[1].set_ylabel('Annual charges ($)')

plt.tight_layout()
plt.show()

# Key business numbers
smoker_avg = df[df['smoker']=='yes']['annual_charges'].mean()
nonsmoker_avg = df[df['smoker']=='no']['annual_charges'].mean()
print(f"Average smoker charge:     ${smoker_avg:>10,.0f}")
print(f"Average non-smoker charge: ${nonsmoker_avg:>10,.0f}")
print(f"Difference:                ${smoker_avg - nonsmoker_avg:>10,.0f}  ({(smoker_avg/nonsmoker_avg - 1)*100:.0f}% more)")

> 💡 **Executive Insight:** The chart on the left shows a **right-skewed distribution** — a few customers cost a lot more than the rest. That long tail is where most underwriting losses come from, and where the most upside exists for better risk selection.

## 3. Interactive Exploration — *Slice the book yourself*

Drag the sliders below to filter the customer base by age and BMI. The chart updates live.

This is what data scientists call **exploratory data analysis (EDA)** — the equivalent of walking the floor before making decisions.

In [ ]:
def explore(age_min, age_max, bmi_max, smoker_filter):
    subset = df[
        (df['age'] >= age_min) &
        (df['age'] <= age_max) &
        (df['bmi'] <= bmi_max)
    ]
    if smoker_filter != 'all':
        subset = subset[subset['smoker'] == smoker_filter]

    if len(subset) == 0:
        print("No customers match these filters. Try widening the range.")
        return

    fig, ax = plt.subplots(figsize=(11, 5))
    colors = subset['smoker'].map({'yes': '#D62246', 'no': '#06A77D'})
    ax.scatter(subset['age'], subset['annual_charges'], c=colors, alpha=0.5, s=30)
    ax.set_xlabel('Age')
    ax.set_ylabel('Annual charges ($)')
    ax.set_title(f"Customer segment: {len(subset):,} policies | Avg charge: ${subset['annual_charges'].mean():,.0f}")

    # Custom legend
    from matplotlib.patches import Patch
    legend_items = [Patch(color='#D62246', label='Smoker'), Patch(color='#06A77D', label='Non-smoker')]
    ax.legend(handles=legend_items, loc='upper left')
    plt.show()

    print(f"📊 Segment size:    {len(subset):,} customers ({len(subset)/len(df)*100:.1f}% of book)")
    print(f"💰 Total premium:  ${subset['annual_charges'].sum():,.0f}")
    print(f"📈 Average charge: ${subset['annual_charges'].mean():,.0f}")

widgets.interact(
    explore,
    age_min=widgets.IntSlider(min=18, max=70, value=18, description='Min age'),
    age_max=widgets.IntSlider(min=20, max=75, value=75, description='Max age'),
    bmi_max=widgets.FloatSlider(min=20, max=50, value=50, step=0.5, description='Max BMI'),
    smoker_filter=widgets.Dropdown(options=['all', 'yes', 'no'], value='all', description='Smoker')
);

> 💡 **Executive Insight:** Try setting **Max BMI = 30** and **Smoker = no**. Notice how dramatically the average charge drops. This is the underwriting team's intuition, made visible. A real-world dashboard does exactly this — at scale, in real time, for any executive who wants to ask a question.

## 4. Finding Drivers — *What moves the needle?*

A **correlation matrix** answers a simple question: "When this number goes up, does that one usually go up too?"
- Values close to **+1** = strong positive relationship
- Values close to **0** = no relationship
- Values close to **-1** = strong negative relationship

In [ ]:
# Convert categorical variables to numeric so we can correlate them
df_numeric = df.copy()
df_numeric['smoker_num'] = (df_numeric['smoker'] == 'yes').astype(int)
df_numeric['sex_num'] = (df_numeric['sex'] == 'male').astype(int)

corr = df_numeric[['age', 'bmi', 'children', 'smoker_num', 'sex_num', 'annual_charges']].corr()

fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1, ax=ax,
            linewidths=1, linecolor='white', cbar_kws={'label': 'Correlation'})
ax.set_title('What drives annual charges?')
plt.tight_layout()
plt.show()

**Reading the bottom row** (correlation with `annual_charges`):
- **Smoking** (~0.78) is by far the strongest driver
- **Age** (~0.30) matters meaningfully
- **BMI** (~0.20) matters somewhat
- **Sex** and **children** barely matter

> 💡 **Executive Insight:** Correlation is the cheapest, fastest analytical tool you have. Before greenlighting an expensive ML project, always ask: "Have we exhausted what correlations can tell us?" Often, 80% of the value is here.

## 5. Predictive Modeling — *What will happen?*

Now we move from describing the past to **predicting the future**.

### The setup
- We give the model **80%** of the data to learn from (the *training set*)
- We hide the remaining **20%** and use it to test how well the model generalizes (the *test set*)
- This is the single most important discipline in machine learning: **never grade your model on data it has already seen.**

In [ ]:
# Prepare the data — convert categories to numbers
X = pd.get_dummies(df.drop('annual_charges', axis=1), drop_first=True)
y = df['annual_charges']

# Hold out 20% as a test set the model will never see during training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {len(X_train):,} customers")
print(f"Test set:     {len(X_test):,} customers (held out — the model has never seen these)")

In [ ]:
# Train two models and compare them

# Model 1: Linear Regression — the classic actuarial workhorse
linear = LinearRegression()
linear.fit(X_train, y_train)
linear_preds = linear.predict(X_test)

# Model 2: Random Forest — a modern ensemble method
forest = RandomForestRegressor(n_estimators=100, random_state=42)
forest.fit(X_train, y_train)
forest_preds = forest.predict(X_test)

# Compare them on the held-out test set
results = pd.DataFrame({
    'Model': ['Linear Regression', 'Random Forest'],
    'Avg error ($)': [mean_absolute_error(y_test, linear_preds), mean_absolute_error(y_test, forest_preds)],
    'R² (variance explained)': [r2_score(y_test, linear_preds), r2_score(y_test, forest_preds)]
}).round(3)

results

**How to read this:**
- **Avg error** = on average, how many dollars off the model's prediction is
- **R²** = the share of variation in charges the model explains (1.0 is perfect, 0.0 is no better than guessing the mean)

> 💡 **Executive Insight:** The Random Forest typically beats Linear Regression here — but at the cost of being harder to explain to a regulator. **The right model depends on the use case.** For pricing (regulated), interpretability often wins. For internal triage or fraud detection, raw accuracy often wins.

## 6. Explaining the Model — *Why did it predict that?*

A model that nobody trusts won't get used. **Feature importance** tells us which inputs the model leans on most heavily.

In [ ]:
importances = pd.Series(forest.feature_importances_, index=X.columns).sort_values()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2E86AB' if v < 0.1 else '#D62246' for v in importances.values]
ax.barh(importances.index, importances.values, color=colors)
ax.set_xlabel('Relative importance')
ax.set_title('What features does the model rely on most?')
plt.tight_layout()
plt.show()

> 💡 **Executive Insight:** Notice that the model "rediscovered" what your underwriters already knew — smoking, age, and BMI dominate. That's actually a *good sign*. When ML aligns with domain expertise, you trust it. When it disagrees, you investigate. **The most valuable models are the ones that surface drivers experts hadn't considered.**

## 7. Live Pricing Tool — *Putting the model in production*

Below is what your underwriting team would see if this model were deployed in your quoting system. Adjust the customer profile and watch the predicted premium update instantly.

In [ ]:
def predict_premium(age, bmi, children, smoker, sex, region):
    # Build a single-row dataframe matching the training format
    profile = pd.DataFrame([{
        'age': age, 'bmi': bmi, 'children': children,
        'sex': sex, 'smoker': smoker, 'region': region
    }])
    profile_encoded = pd.get_dummies(profile)
    # Align columns with what the model was trained on
    profile_encoded = profile_encoded.reindex(columns=X.columns, fill_value=0)

    pred = forest.predict(profile_encoded)[0]
    book_avg = df['annual_charges'].mean()

    # Display result
    color = '#D62246' if pred > book_avg * 1.5 else ('#F4A261' if pred > book_avg else '#06A77D')
    risk_label = 'High risk' if pred > book_avg * 1.5 else ('Standard' if pred > book_avg else 'Preferred')

    html = f'''
    <div style="border-left: 6px solid {color}; padding: 16px 24px; background: #f8f9fa; margin: 12px 0; font-family: -apple-system, sans-serif;">
        <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px;">Predicted Annual Premium</div>
        <div style="font-size: 32px; font-weight: 700; color: {color};">${pred:,.0f}</div>
        <div style="font-size: 14px; color: #333; margin-top: 6px;">
            Risk tier: <strong>{risk_label}</strong> &nbsp;·&nbsp;
            {(pred / book_avg - 1) * 100:+.0f}% vs. book average (${book_avg:,.0f})
        </div>
    </div>
    '''
    display(HTML(html))

widgets.interact(
    predict_premium,
    age=widgets.IntSlider(min=18, max=75, value=40, description='Age'),
    bmi=widgets.FloatSlider(min=16, max=50, value=27, step=0.5, description='BMI'),
    children=widgets.IntSlider(min=0, max=5, value=1, description='Children'),
    smoker=widgets.Dropdown(options=['yes', 'no'], value='no', description='Smoker'),
    sex=widgets.Dropdown(options=['male', 'female'], value='male', description='Sex'),
    region=widgets.Dropdown(options=['northeast', 'southeast', 'southwest', 'northwest'], value='northeast', description='Region')
);

> 💡 **Executive Insight:** This is the *whole point* of a data science project — taking an insight that lived in a spreadsheet and embedding it into a tool that hundreds of underwriters use every day. The technical work is one part. The integration into the workflow is what creates dollars.

## 8. Customer Segmentation — *Discovering hidden groups*

So far we've predicted a known quantity (price). But sometimes you want to **discover structure** in your customer base that you didn't know was there.

This is **unsupervised learning**. We'll use an algorithm called **K-Means** to group customers into 4 natural segments based on age, BMI, and charges.

In [ ]:
# Pick the features for segmentation and standardize them
features = df[['age', 'bmi', 'annual_charges']].copy()
scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

# Run K-Means
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['segment'] = kmeans.fit_predict(features_scaled)

# Profile each segment in plain English
segment_profile = df.groupby('segment').agg(
    customers=('age', 'size'),
    avg_age=('age', 'mean'),
    avg_bmi=('bmi', 'mean'),
    pct_smokers=('smoker', lambda s: (s == 'yes').mean() * 100),
    avg_charges=('annual_charges', 'mean'),
    total_premium=('annual_charges', 'sum')
).round(1).sort_values('avg_charges')

# Give them human-friendly names based on charge level (sorted from low to high)
n_segments = len(segment_profile)
default_names = ['Healthy & Young', 'Standard Adult', 'Older / Higher BMI', 'High-Risk (smokers)']
segment_profile['name'] = default_names[:n_segments]
segment_profile = segment_profile[['name', 'customers', 'avg_age', 'avg_bmi', 'pct_smokers', 'avg_charges', 'total_premium']]
segment_profile.columns = ['Segment', 'Customers', 'Avg age', 'Avg BMI', '% smokers', 'Avg charge ($)', 'Total premium ($)']
segment_profile

In [ ]:
# Visualize the segments
fig, ax = plt.subplots(figsize=(11, 6))
palette = ['#06A77D', '#2E86AB', '#F4A261', '#D62246']

for seg_id in sorted(df['segment'].unique()):
    sub = df[df['segment'] == seg_id]
    ax.scatter(sub['age'], sub['annual_charges'],
               c=palette[seg_id], label=f"Segment {seg_id}", alpha=0.5, s=25)

ax.set_xlabel('Age')
ax.set_ylabel('Annual charges ($)')
ax.set_title('Customer segments discovered by the algorithm')
ax.legend(title='Segment')
plt.tight_layout()
plt.show()

> 💡 **Executive Insight:** The algorithm wasn't told what to look for — it found these groups on its own. In practice, segmentation drives:
> - **Targeted marketing** (different messages for different segments)
> - **Wellness program enrollment** (focus on segments with the highest preventable risk)
> - **Retention strategies** (which segments are at risk of leaving?)
> - **Reinsurance decisions** (cede more of the high-risk segment)

## 9. Quantifying the Business Value

The technical work means nothing without dollars attached. Let's estimate the value of moving from "rule-of-thumb pricing" to "model-based pricing".

In [ ]:
# Baseline: charge everyone the average
naive_pricing_error = mean_absolute_error(y_test, [y_train.mean()] * len(y_test))

# Model-based pricing
model_pricing_error = mean_absolute_error(y_test, forest_preds)

improvement_per_policy = naive_pricing_error - model_pricing_error

# Realistically, only a fraction of pricing-error reduction translates into margin
# (competition, regulation, and customer behavior absorb most of the rest)
realization_rate = 0.05  # capture 5% of the theoretical improvement
book_size = 100_000      # hypothetical mid-size book

annual_value = improvement_per_policy * realization_rate * book_size

print(f"Average pricing error WITHOUT a model:   ${naive_pricing_error:>10,.0f} per policy")
print(f"Average pricing error WITH the model:    ${model_pricing_error:>10,.0f} per policy")
print(f"Theoretical improvement per policy:      ${improvement_per_policy:>10,.0f}")
print()
print(f"Assumptions for the business case:")
print(f"  Book size:                             {book_size:>12,} policies")
print(f"  Realization rate (post-competition):   {realization_rate:>12.0%}")
print()
print(f"  Estimated annual margin uplift:        ${annual_value:>14,.0f}")
print()
print("This is a deliberately conservative back-of-envelope. Real ROI work")
print("models adverse selection, churn, regulatory constraints, and competitive")
print("response — but the *order of magnitude* is what matters for a Go/No-Go decision.")

> 💡 **Executive Insight:** This kind of back-of-envelope calculation is what gets data science projects funded. The framing should always be:
> 1. What decision are we improving?
> 2. How much is each decision worth?
> 3. How many decisions do we make per year?
>
> If you can't answer those three questions, the project isn't ready to start.

## 10. The Big Picture — Where this fits in your business

You've now seen, end-to-end, the workflow that powers most modern insurance analytics:

| Stage | Question | What you saw |
|---|---|---|
| **Descriptive** | What happened? | Section 2 — distributions, averages |
| **Diagnostic** | Why did it happen? | Section 4 — correlations |
| **Predictive** | What will happen? | Sections 5–7 — the model & calculator |
| **Prescriptive** | What should we do? | Sections 8–9 — segments & ROI |

### Where the value really lives
The technical work is rarely the bottleneck. The hard parts are:
1. **Data quality** — most projects spend 60–80% of their time here
2. **Stakeholder alignment** — picking *which* decision to improve
3. **Productionization** — getting the model into the workflow
4. **Governance** — fairness, explainability, regulatory compliance

---

*🎉 Congratulations — you've completed a working data science project. The same toolkit, scaled up to your real data, drives most of the analytical innovation in the industry today.*